# 3D fermionic toric code — $h_z$ phase-diagram sweep (self-contained)

Matrix-free Krylov diagonalization of the **3D fermionic toric code** at $L=2$
PBC ($N = 3\cdot2^3 = 24$ qubits, $2^{24}\approx16.8$M states).  Everything is
inlined — **no `netket`, no `jax`, no repo imports** — only `numpy`, `scipy`,
`numba`.

The decorated plaquette stabilizer is
$$\tilde B_p = \Big(\prod_{e\in\partial p}\sigma^z_e\Big)\,\sigma^x_{e_+}\,\sigma^x_{e_-},$$
with the two $\sigma^x$ on opposite transverse corner edges (a body diagonal:
the $(+a,+b)$ corner edge on the $+$perp side and the $(-a,-b)$ corner edge on
the $-$perp side).  Vertex stars $A_v$ are the usual all-$\sigma^x$ stars.

**Order-parameter caveat.**  The bare $\sigma^z$ z-line wrapping the torus
anticommutes with 4 of the decorated plaquettes, so it is *not* a conserved
Wilson string here.  We keep it as the BFFM diagnostic for now and also plot the
spectral gap and $\langle M_z\rangle$, which are well defined.

> **Runtime.**  Colab CPU runtime is enough (eigsh is CPU-only).  At $N=24$ a
> single `eigsh` needs ~2.7 GB of Lanczos workspace; the free tier (~12 GB RAM)
> handles it.  A 20-point sweep takes on the order of tens of minutes.

In [ ]:
# Colab already ships numpy/scipy/numba; this is a no-op if they're present.
!pip install -q numba

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Tuple
from numba import njit, prange
from scipy.sparse.linalg import LinearOperator, eigsh

## Geometry (netket-free 3D toric code, PBC)

In [ ]:
class ThreeDToricPBC:
    """Minimal 3D toric-code geometry on an L^3 cubic lattice with PBC.

    Qubits live on edges.  An edge along axis d at integer vertex (x,y,z) has
    midpoint (x,y,z)+0.5 e_d; it is keyed by its integer 2x-coordinate tuple
    (avoids float-equality issues).  Exposes the interface the Hamiltonian and
    decoration code consume: N, Lx/Ly/Lz, _coord_to_idx, vertex_all, plaq_all.
    """
    def __init__(self, L: int):
        self.Lx = self.Ly = self.Lz = L
        self.N = 3 * L**3
        self._coord_to_idx = {}
        idx = 0
        for d in range(3):
            for x in range(L):
                for y in range(L):
                    for z in range(L):
                        key = (2*x + (d == 0), 2*y + (d == 1), 2*z + (d == 2))
                        self._coord_to_idx[key] = idx
                        idx += 1
        self.vertex_all = self._stars()
        self.plaq_all = self._plaqs()

    def _idx(self, kx, ky, kz):
        return self._coord_to_idx[(kx % (2*self.Lx), ky % (2*self.Ly), kz % (2*self.Lz))]

    def _stars(self):
        out = []
        for x in range(self.Lx):
            for y in range(self.Ly):
                for z in range(self.Lz):
                    out.append([
                        self._idx(2*x+1, 2*y, 2*z), self._idx(2*x-1, 2*y, 2*z),
                        self._idx(2*x, 2*y+1, 2*z), self._idx(2*x, 2*y-1, 2*z),
                        self._idx(2*x, 2*y, 2*z+1), self._idx(2*x, 2*y, 2*z-1),
                    ])
        return out

    def _plaqs(self):
        out = []
        for c in range(3):                       # normal axis
            a, b = [i for i in range(3) if i != c]
            for x in range(self.Lx):
                for y in range(self.Ly):
                    for z in range(self.Lz):
                        center = [2*x, 2*y, 2*z]
                        center[a] += 1; center[b] += 1
                        edges = []
                        for axis, s in [(a, +1), (a, -1), (b, +1), (b, -1)]:
                            k = list(center); k[axis] += s
                            edges.append(self._idx(*k))
                        out.append(edges)
        return out

## Matrix-free Hamiltonian and Pauli-string expectations

In [ ]:
@njit(inline='always')
def _bit_parity64(x):
    x ^= x >> 32; x ^= x >> 16; x ^= x >> 8
    x ^= x >> 4;  x ^= x >> 2;  x ^= x >> 1
    return x & 1


@njit(parallel=True, fastmath=True)
def _matvec_jit(psi, diag, x_masks, x_coefs, out):
    dim = psi.shape[0]; n_masks = x_masks.shape[0]
    for i in prange(dim):
        s = diag[i] * psi[i]
        for k in range(n_masks):
            s += x_coefs[k] * psi[i ^ x_masks[k]]
        out[i] = s


@njit(parallel=True, fastmath=True)
def _matvec_jit_xz_add(psi, xz_z_masks, xz_x_masks, xz_coefs, out):
    dim = psi.shape[0]; n = xz_z_masks.shape[0]
    for j in prange(dim):
        s = 0.0
        for k in range(n):
            sign = 1.0 - 2.0 * _bit_parity64(j & xz_z_masks[k])
            s += xz_coefs[k] * sign * psi[j ^ xz_x_masks[k]]
        out[j] += s


def qubits_to_mask(indices):
    m = 0
    for i in indices:
        if i == -1:
            continue
        m |= (1 << int(i))
    return m


def z_string_eigvals(basis, mask, N):
    parity = np.zeros_like(basis, dtype=np.int8)
    for i in range(N):
        if mask & (1 << i):
            parity ^= ((basis >> i) & 1).astype(np.int8)
    return 1.0 - 2.0 * parity.astype(np.float64)


def hamiltonian_linop(geom, hx=0.0, hy=0.0, hz=0.0, J=1.0, xz_stabs=None, dtype=np.float64):
    """Matrix-free toric-code Hamiltonian -> (LinearOperator, basis).

    xz_stabs: optional list of (z_qubits, x_qubits, coef) generalized stabilizers.
    When non-empty they REPLACE the default Z-only plaquette term.
    """
    if hy != 0:
        raise NotImplementedError("Y-field not supported (h_y = 0 assumed).")
    N = geom.N
    dim = 1 << N
    basis = np.arange(dim, dtype=np.int64)
    use_xz = xz_stabs is not None and len(xz_stabs) > 0

    diag = np.zeros(dim, dtype=dtype)
    if not use_xz:
        for p in geom.plaq_all:
            diag -= J * z_string_eigvals(basis, qubits_to_mask(p), N)
    if hz != 0.0:
        for i in range(N):
            diag -= hz * z_string_eigvals(basis, 1 << i, N)

    x_strings = defaultdict(float)
    for v in geom.vertex_all:
        x_strings[qubits_to_mask(v)] -= J
    if hx != 0.0:
        for i in range(N):
            x_strings[1 << i] -= hx

    xz_z_list, xz_x_list, xz_c_list = [], [], []
    if use_xz:
        for z_qubits, x_qubits, coef in xz_stabs:
            zm = qubits_to_mask(z_qubits); xm = qubits_to_mask(x_qubits)
            if zm & xm:
                raise ValueError(f"overlapping z and x supports: z={z_qubits}, x={x_qubits}")
            if xm == 0:
                diag += coef * z_string_eigvals(basis, zm, N)
            elif zm == 0:
                x_strings[xm] += coef
            else:
                xz_z_list.append(zm); xz_x_list.append(xm); xz_c_list.append(coef)

    x_strings = {m: c for m, c in x_strings.items() if c != 0.0 and m != 0}
    if x_strings:
        x_masks = np.fromiter(x_strings.keys(), dtype=np.int64, count=len(x_strings))
        x_coefs = np.fromiter(x_strings.values(), dtype=dtype, count=len(x_strings))
    else:
        x_masks = np.empty(0, dtype=np.int64); x_coefs = np.empty(0, dtype=dtype)

    xz_z_masks = np.array(xz_z_list, dtype=np.int64) if xz_z_list else np.empty(0, dtype=np.int64)
    xz_x_masks = np.array(xz_x_list, dtype=np.int64) if xz_x_list else np.empty(0, dtype=np.int64)
    xz_coefs   = np.array(xz_c_list, dtype=dtype)    if xz_c_list else np.empty(0, dtype=dtype)
    has_xz = xz_z_masks.size > 0

    def matvec(psi):
        psi = np.ascontiguousarray(psi, dtype=dtype)
        out = np.empty_like(psi)
        _matvec_jit(psi, diag, x_masks, x_coefs, out)
        if has_xz:
            _matvec_jit_xz_add(psi, xz_z_masks, xz_x_masks, xz_coefs, out)
        return out

    return LinearOperator((dim, dim), matvec=matvec, dtype=dtype), basis


def expect_x_string(psi, basis, mask):
    if mask == 0:
        return float(np.real(np.sum(np.abs(psi) ** 2)))
    return float(np.real(np.sum(np.conj(psi) * psi[basis ^ mask])))


def expect_z_string(psi, basis, mask, N):
    if mask == 0:
        return float(np.sum(np.abs(psi) ** 2))
    return float(np.sum(np.abs(psi) ** 2 * z_string_eigvals(basis, mask, N)))


def expect_xz_string(psi, basis, z_mask, x_mask, N):
    if x_mask == 0:
        return expect_z_string(psi, basis, z_mask, N)
    if z_mask == 0:
        return expect_x_string(psi, basis, x_mask)
    signs = z_string_eigvals(basis, z_mask, N)
    return float(np.real(np.sum(np.conj(psi) * signs * psi[basis ^ x_mask])))

## Fermionic plaquette decoration

In [ ]:
ORIENTATIONS = ("xy", "yz", "zx")
_ORIENT_INFO = {
    "xy": (np.array([0.5, 0.5, 0.0]), 2),
    "yz": (np.array([0.0, 0.5, 0.5]), 0),
    "zx": (np.array([0.5, 0.0, 0.5]), 1),
}


@dataclass(frozen=True)
class Plaquette:
    ix: int; iy: int; iz: int
    orientation: str
    boundary_edges: Tuple[int, int, int, int]

    @property
    def center(self):
        offset, _ = _ORIENT_INFO[self.orientation]
        return np.array([self.ix, self.iy, self.iz], dtype=float) + offset

    @property
    def perp_axis(self):
        return _ORIENT_INFO[self.orientation][1]

    @property
    def in_plane_axes(self):
        perp = self.perp_axis
        return tuple(a for a in range(3) if a != perp)


def _coord_to_idx_pbc(geom, coord):
    L = (geom.Lx, geom.Ly, geom.Lz)
    key = tuple(int(round(2 * coord[d])) % (2 * int(L[d])) for d in range(3))
    return int(geom._coord_to_idx[key])


def enumerate_plaquettes(geom):
    out = []
    for orient in ORIENTATIONS:
        offset, perp = _ORIENT_INFO[orient]
        a, b = tuple(d for d in range(3) if d != perp)
        for ix in range(geom.Lx):
            for iy in range(geom.Ly):
                for iz in range(geom.Lz):
                    center = np.array([ix, iy, iz], dtype=float) + offset
                    edges = []
                    for axis, sign in [(a, +0.5), (a, -0.5), (b, +0.5), (b, -0.5)]:
                        c = center.copy(); c[axis] += sign
                        edges.append(_coord_to_idx_pbc(geom, c))
                    out.append(Plaquette(ix, iy, iz, orient, tuple(edges)))
    return out


def decoration_edges(geom, p):
    """Two transverse sigma^x edges: (+a,+b) corner on +perp side, (-a,-b) on -perp."""
    perp = p.perp_axis; a, b = p.in_plane_axes
    out = []
    for da, db, s in [(+0.5, +0.5, +1), (-0.5, -0.5, -1)]:
        c = p.center.copy(); c[a] += da; c[b] += db; c[perp] += s * 0.5
        out.append(_coord_to_idx_pbc(geom, c))
    return out


def fermionic_plaquettes(geom, J=1.0):
    """(z_qubits, x_qubits, coef) list, drop-in for hamiltonian_linop(xz_stabs=...)."""
    out = []
    for p in enumerate_plaquettes(geom):
        out.append((list(p.boundary_edges), decoration_edges(geom, p), -float(J)))
    return out


def _anticommute(z1, x1, z2, x2):
    return ((bin(z1 & x2).count("1") + bin(z2 & x1).count("1")) & 1) != 0


def verify_xz_commutation(stabs, vertex_all):
    entries = [(qubits_to_mask(z), qubits_to_mask(x)) for z, x, _ in stabs]
    entries += [(0, qubits_to_mask(v)) for v in vertex_all]
    violations = []
    n = len(entries)
    for i in range(n):
        z1, x1 = entries[i]
        for j in range(i + 1, n):
            z2, x2 = entries[j]
            if _anticommute(z1, x1, z2, x2):
                violations.append((i, j))
    return {"ok": len(violations) == 0, "violations": violations, "n_stabilizers": n}

## Setup + zero-field sanity check

In [ ]:
def z_line_qubits(geom, x=0, y=0):
    """sigma^z edges along z at column (x, y), wrapping the torus (PBC)."""
    Lx2, Ly2, Lz2 = 2*geom.Lx, 2*geom.Ly, 2*geom.Lz
    return [geom._coord_to_idx[((2*x) % Lx2, (2*y) % Ly2, (2*z + 1) % Lz2)]
            for z in range(geom.Lz)]


geom = ThreeDToricPBC(2)
stabs_f = fermionic_plaquettes(geom, J=1.0)
chk = verify_xz_commutation(stabs_f, geom.vertex_all)
print(f"fermionic 3D TC L=2 PBC:  N = {geom.N},  |A_v| = {len(geom.vertex_all)},  "
      f"|B~_p| = {len(stabs_f)},  commute = {chk['ok']}  ({len(chk['violations'])} violations)")

# Zero-field sanity: E0 = -(N_v + N_p), every decorated stabilizer +1.
H, basis = hamiltonian_linop(geom, hx=0.0, hz=0.0, J=1.0, xz_stabs=stabs_f)
ev, vc = eigsh(H, k=4, which="SA", tol=1e-7)
psi = vc[:, 0]
A = np.mean([expect_x_string(psi, basis, qubits_to_mask(v)) for v in geom.vertex_all])
B = np.mean([expect_xz_string(psi, basis, qubits_to_mask(z), qubits_to_mask(x), geom.N)
             for z, x, _ in stabs_f])
print(f"eigenvalues: {ev}")
print(f"mean <A_v>  = {A:.4f}    (expect 1.0)")
print(f"mean <B~_p> = {B:.4f}    (expect 1.0)")
print(f"E0 = {ev[0]:.4f}        (expect -{len(geom.vertex_all) + len(stabs_f)})")

## $h_z$ sweep

In [ ]:
def sweep_phase_diagram_3d_fermionic(geom, stabs, h_z_list, hx=0.3, k=2,
                                     label="3D fermionic L=2 PBC"):
    N = geom.N
    basis = np.arange(1 << N, dtype=np.int64)
    cz = z_line_qubits(geom, 0, 0)
    oz = cz[: max(1, len(cz) // 2)]
    cz_mask, oz_mask = qubits_to_mask(cz), qubits_to_mask(oz)
    star_masks = [qubits_to_mask(v) for v in geom.vertex_all]
    plaq_zx = [(qubits_to_mask(z), qubits_to_mask(x)) for z, x, _ in stabs]
    site_masks = [1 << i for i in range(N)]

    keys = ("h_z", "E0", "gap", "A", "B", "Mz", "Wz", "Oz", "bffm_z")
    rec = {kk: [] for kk in keys}
    for h_z in tqdm(h_z_list, desc=label):
        H, _ = hamiltonian_linop(geom, hx=hx, hz=float(h_z), J=1.0, xz_stabs=stabs)
        ev, vc = eigsh(H, k=k, which="SA", tol=1e-7, return_eigenvectors=True)
        psi = vc[:, 0]
        Wz = expect_z_string(psi, basis, cz_mask, N)
        Oz = expect_z_string(psi, basis, oz_mask, N)
        rec["h_z"].append(float(h_z))
        rec["E0"].append(float(ev[0]))
        rec["gap"].append(float(ev[1] - ev[0]))
        rec["A"].append(float(np.mean([expect_x_string(psi, basis, m) for m in star_masks])))
        rec["B"].append(float(np.mean([expect_xz_string(psi, basis, zm, xm, N) for zm, xm in plaq_zx])))
        rec["Mz"].append(float(np.mean([expect_z_string(psi, basis, m, N) for m in site_masks])))
        rec["Wz"].append(Wz); rec["Oz"].append(Oz)
        rec["bffm_z"].append(Oz / np.sqrt(abs(Wz)) if abs(Wz) > 1e-14 else np.nan)
    return {kk: np.array(v) for kk, v in rec.items()}


h_z_list = np.linspace(0, 0.75, 20)
rec_3df = sweep_phase_diagram_3d_fermionic(geom, stabs_f, h_z_list, hx=0.3)

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rec_3df["h_z"], rec_3df["bffm_z"], "o-", color="C3", label="3D fermionic L=2 PBC")
axes[1].plot(rec_3df["h_z"], np.gradient(rec_3df["bffm_z"], rec_3df["h_z"]),
             "o-", color="C3", label="3D fermionic L=2 PBC")
axes[0].set_ylabel(r"$O_{BFFM}^Z$")
axes[1].set_ylabel(r"$\partial O_{BFFM}^Z / \partial h_z$")
for ax in axes:
    ax.set_xlabel(r"$h_z$"); ax.legend()
fig.suptitle(r"3D fermionic TC, $h_x = 0.3$, bare $\sigma^z$ z-line")
fig.tight_layout(); plt.show()

# Well-defined transition diagnostics for the fermionic model.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rec_3df["h_z"], rec_3df["gap"], "o-", color="C3")
axes[1].plot(rec_3df["h_z"], rec_3df["Mz"], "o-", color="C3")
axes[0].set_ylabel("spectral gap"); axes[1].set_ylabel(r"$\langle M_z\rangle$")
for ax in axes:
    ax.set_xlabel(r"$h_z$")
fig.suptitle(r"3D fermionic TC, $h_x = 0.3$: gap and magnetization")
fig.tight_layout(); plt.show()